In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import py3Dmol

In [2]:
def find_project_root(starting_directory=None):
    """
    Search upward from the current directory until a folder
    containing 'xyz_files' is found.
    """
    start = (
        Path(starting_directory).resolve()
        if starting_directory
        else Path.cwd().resolve()
    )

    possible_directories = [start, *start.parents]

    for directory in possible_directories:
        xyz_directory = directory / "xyz_files"

        if xyz_directory.is_dir():
            return directory

    raise FileNotFoundError(
        "Could not find a project folder containing 'xyz_files'. "
        "Open the GSCDB Benchmarking folder in VS Code and run again."
    )


PROJECT_ROOT = find_project_root()
XYZ_DIR = PROJECT_ROOT / "xyz_files"

ethene_path = XYZ_DIR / "DARC_ethene.xyz"
butadiene_path = XYZ_DIR / "DARC_butadiene.xyz"
product_path = XYZ_DIR / "DARC_P1.xyz"

print("Project root:", PROJECT_ROOT)
print("XYZ directory:", XYZ_DIR)

print("\nFile checks:")
print("Ethene exists:", ethene_path.exists())
print("Butadiene exists:", butadiene_path.exists())
print("Product exists:", product_path.exists())

Project root: C:\Users\91988\GSCDB Benchmarking
XYZ directory: C:\Users\91988\GSCDB Benchmarking\xyz_files

File checks:
Ethene exists: True
Butadiene exists: True
Product exists: True


In [3]:
def read_xyz_file(file_path):
    """
    Read and validate an XYZ molecular-structure file.

    Returns
    -------
    xyz_text : str
        Full XYZ content for py3Dmol.

    atoms : list of dict
        Atomic symbols and Cartesian coordinates.

    comment : str
        Metadata from the second line of the XYZ file.
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"XYZ file does not exist: {file_path}"
        )

    xyz_text = file_path.read_text(encoding="utf-8")
    lines = xyz_text.splitlines()

    if len(lines) < 3:
        raise ValueError(
            f"{file_path.name} does not appear to be a valid XYZ file."
        )

    try:
        number_of_atoms = int(lines[0].strip())
    except ValueError as error:
        raise ValueError(
            f"The first line of {file_path.name} is not an atom count."
        ) from error

    comment = lines[1].strip()
    coordinate_lines = lines[2 : 2 + number_of_atoms]

    if len(coordinate_lines) != number_of_atoms:
        raise ValueError(
            f"{file_path.name}: expected {number_of_atoms} coordinate "
            f"lines but found {len(coordinate_lines)}."
        )

    atoms = []

    for atom_number, line in enumerate(
        coordinate_lines,
        start=1
    ):
        parts = line.split()

        if len(parts) < 4:
            raise ValueError(
                f"Invalid coordinate line for atom {atom_number}: {line}"
            )

        element = parts[0]

        try:
            x, y, z = map(float, parts[1:4])
        except ValueError as error:
            raise ValueError(
                f"Invalid coordinates for atom {atom_number}: {line}"
            ) from error

        atoms.append(
            {
                "atom_number": atom_number,
                "element": element,
                "x": x,
                "y": y,
                "z": z,
            }
        )

    return xyz_text, atoms, comment

In [4]:
ethene_xyz, ethene_atoms, ethene_comment = read_xyz_file(
    ethene_path
)

butadiene_xyz, butadiene_atoms, butadiene_comment = read_xyz_file(
    butadiene_path
)

product_xyz, product_atoms, product_comment = read_xyz_file(
    product_path
)

In [5]:
print("Ethene atoms:", len(ethene_atoms))
print("Butadiene atoms:", len(butadiene_atoms))
print("Product atoms:", len(product_atoms))

print("\nEthene metadata:")
print(ethene_comment)

print("\nButadiene metadata:")
print(butadiene_comment)

print("\nProduct metadata:")
print(product_comment)

Ethene atoms: 6
Butadiene atoms: 10
Product atoms: 16

Ethene metadata:
charge=0, multiplicity=1, basis=def2-QZVPPD, AUX_BASIS_CORR=rimp2-def2-QZVPPD, SCF_ALGORITHM=GDM, xc_grid=000099000590, mem_total=3750, num_basis=258, num_pairs=33789, num_threads=1

Butadiene metadata:
charge=0, multiplicity=1, basis=def2-QZVPPD, AUX_BASIS_CORR=rimp2-def2-QZVPPD, SCF_ALGORITHM=GDM, xc_grid=000099000590, mem_total=3750, num_basis=450, num_pairs=96270, num_threads=1

Product metadata:
charge=0, multiplicity=1, basis=def2-QZVPPD, mem_total=7500, AUX_BASIS_CORR=rimp2-def2-QZVPPD, SCF_ALGORITHM=GDM, xc_grid=000099000590, num_basis=708, num_pairs=238496, num_threads=2


In [6]:
def molecular_formula(atoms):
    counts = Counter(
        atom["element"] for atom in atoms
    )

    conventional_order = ["C", "H", "N", "O", "F", "P", "S", "Cl", "Br"]

    formula_parts = []

    for element in conventional_order:
        if element in counts:
            count = counts.pop(element)

            formula_parts.append(
                element if count == 1 else f"{element}{count}"
            )

    for element in sorted(counts):
        count = counts[element]

        formula_parts.append(
            element if count == 1 else f"{element}{count}"
        )

    return "".join(formula_parts)


print("Ethene formula:", molecular_formula(ethene_atoms))
print("Butadiene formula:", molecular_formula(butadiene_atoms))
print("Product formula:", molecular_formula(product_atoms))

Ethene formula: C2H4
Butadiene formula: C4H6
Product formula: C6H10


In [7]:
def visualise_xyz(
    xyz_text,
    width=600,
    height=450,
    background="white"
):
    viewer = py3Dmol.view(
        width=width,
        height=height
    )

    viewer.addModel(
        xyz_text,
        "xyz"
    )

    viewer.setStyle(
        {},
        {
            "stick": {
                "radius": 0.15
            },
            "sphere": {
                "scale": 0.28
            }
        }
    )

    viewer.setBackgroundColor(background)
    viewer.zoomTo()

    return viewer

In [8]:
ethene_view = visualise_xyz(
    ethene_xyz
)

ethene_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
butadiene_view = visualise_xyz(
    butadiene_xyz
)

butadiene_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
product_view = visualise_xyz(
    product_xyz
)

product_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [11]:
reaction_viewer = py3Dmol.view(
    width=1200,
    height=450,
    viewergrid=(1, 3),
    linked=False
)

structures = [
    (
        ethene_xyz,
        "DARC_ethene",
        (0, 0)
    ),
    (
        butadiene_xyz,
        "DARC_butadiene",
        (0, 1)
    ),
    (
        product_xyz,
        "DARC_P1",
        (0, 2)
    ),
]

for xyz_text, structure_name, panel in structures:
    reaction_viewer.addModel(
        xyz_text,
        "xyz",
        viewer=panel
    )

    reaction_viewer.setStyle(
        {},
        {
            "stick": {
                "radius": 0.14
            },
            "sphere": {
                "scale": 0.27
            }
        },
        viewer=panel
    )

    reaction_viewer.setBackgroundColor(
        "white",
        viewer=panel
    )

    reaction_viewer.zoomTo(
        viewer=panel
    )

reaction_viewer

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [12]:
def atom_position(atoms, atom_number):
    """
    Return the Cartesian position of a one-based atom number.
    """
    if atom_number < 1 or atom_number > len(atoms):
        raise IndexError(
            f"Atom number must be between 1 and {len(atoms)}."
        )

    atom = atoms[atom_number - 1]

    return np.array(
        [
            atom["x"],
            atom["y"],
            atom["z"],
        ],
        dtype=float
    )


def distance_between_atoms(
    atoms,
    atom_number_1,
    atom_number_2
):
    position_1 = atom_position(
        atoms,
        atom_number_1
    )

    position_2 = atom_position(
        atoms,
        atom_number_2
    )

    return float(
        np.linalg.norm(position_1 - position_2)
    )

In [13]:
ethene_cc = distance_between_atoms(
    ethene_atoms,
    1,
    2
)

print(
    f"Ethene C1=C2 distance: {ethene_cc:.3f} Å"
)

Ethene C1=C2 distance: 1.323 Å


In [14]:
butadiene_c1_c2 = distance_between_atoms(
    butadiene_atoms,
    1,
    2
)

butadiene_c2_c3 = distance_between_atoms(
    butadiene_atoms,
    2,
    3
)

butadiene_c3_c4 = distance_between_atoms(
    butadiene_atoms,
    3,
    4
)

print(
    f"Butadiene C1=C2: {butadiene_c1_c2:.3f} Å"
)

print(
    f"Butadiene C2-C3: {butadiene_c2_c3:.3f} Å"
)

print(
    f"Butadiene C3=C4: {butadiene_c3_c4:.3f} Å"
)

Butadiene C1=C2: 1.332 Å
Butadiene C2-C3: 1.449 Å
Butadiene C3=C4: 1.332 Å


In [15]:
product_ring_bonds = [
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 5),
    (5, 6),
    (6, 1),
]

for atom_1, atom_2 in product_ring_bonds:
    bond_length = distance_between_atoms(
        product_atoms,
        atom_1,
        atom_2
    )

    print(
        f"Product C{atom_1}-C{atom_2}: "
        f"{bond_length:.3f} Å"
    )

Product C1-C2: 1.498 Å
Product C2-C3: 1.524 Å
Product C3-C4: 1.522 Å
Product C4-C5: 1.524 Å
Product C5-C6: 1.498 Å
Product C6-C1: 1.329 Å


In [16]:
def coordinate_dictionary(atoms, atom_number):
    atom = atoms[atom_number - 1]

    return {
        "x": atom["x"],
        "y": atom["y"],
        "z": atom["z"],
    }


def midpoint_dictionary(
    atoms,
    atom_number_1,
    atom_number_2
):
    point_1 = atom_position(
        atoms,
        atom_number_1
    )

    point_2 = atom_position(
        atoms,
        atom_number_2
    )

    midpoint = (
        point_1 + point_2
    ) / 2.0

    return {
        "x": float(midpoint[0]),
        "y": float(midpoint[1]),
        "z": float(midpoint[2]),
    }

In [17]:
product_highlight_view = py3Dmol.view(
    width=700,
    height=550
)

product_highlight_view.addModel(
    product_xyz,
    "xyz"
)

product_highlight_view.setStyle(
    {},
    {
        "stick": {
            "radius": 0.14
        },
        "sphere": {
            "scale": 0.27
        }
    }
)

new_bonds = [
    (2, 3),
    (4, 5),
]

for atom_1, atom_2 in new_bonds:
    product_highlight_view.addCylinder(
        {
            "start": coordinate_dictionary(
                product_atoms,
                atom_1
            ),
            "end": coordinate_dictionary(
                product_atoms,
                atom_2
            ),
            "radius": 0.055,
            "color": "red",
            "opacity": 0.85,
            "fromCap": True,
            "toCap": True,
        }
    )

    bond_length = distance_between_atoms(
        product_atoms,
        atom_1,
        atom_2
    )

    product_highlight_view.addLabel(
        f"New C{atom_1}-C{atom_2}: "
        f"{bond_length:.3f} Å",
        {
            "position": midpoint_dictionary(
                product_atoms,
                atom_1,
                atom_2
            ),
            "backgroundColor": "white",
            "fontColor": "red",
            "fontSize": 14,
            "showBackground": True,
        }
    )

product_highlight_view.setBackgroundColor("white")
product_highlight_view.zoomTo()

product_highlight_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [18]:
product_highlight_view.addCylinder(
    {
        "start": coordinate_dictionary(
            product_atoms,
            1
        ),
        "end": coordinate_dictionary(
            product_atoms,
            6
        ),
        "radius": 0.06,
        "color": "blue",
        "opacity": 0.85,
        "fromCap": True,
        "toCap": True,
    }
)

remaining_double_bond = distance_between_atoms(
    product_atoms,
    1,
    6
)

product_highlight_view.addLabel(
    f"Remaining C1=C6: "
    f"{remaining_double_bond:.3f} Å",
    {
        "position": midpoint_dictionary(
            product_atoms,
            1,
            6
        ),
        "backgroundColor": "white",
        "fontColor": "blue",
        "fontSize": 14,
        "showBackground": True,
    }
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [20]:
for carbon_number in range(1, 7):
    product_highlight_view.addLabel(
        f"C{carbon_number}",
        {
            "position": coordinate_dictionary(
                product_atoms,
                carbon_number
            ),
            "backgroundColor": "white",
            "fontColor": "black",
            "fontSize": 13,
            "showBackground": True,
        }
    )

In [21]:
def translated_xyz(
    atoms,
    comment,
    translation
):
    """
    Create a new XYZ string after translating all atoms.

    Parameters
    ----------
    atoms : list of dict
        Parsed atoms.

    comment : str
        XYZ comment line.

    translation : tuple
        Translation in Å: (dx, dy, dz).
    """
    dx, dy, dz = translation

    lines = [
        str(len(atoms)),
        comment,
    ]

    for atom in atoms:
        x = atom["x"] + dx
        y = atom["y"] + dy
        z = atom["z"] + dz

        lines.append(
            f"{atom['element']:2s} "
            f"{x:15.8f} "
            f"{y:15.8f} "
            f"{z:15.8f}"
        )

    return "\n".join(lines)

In [23]:
butadiene_display_xyz = translated_xyz(
    butadiene_atoms,
    butadiene_comment,
    translation=(-2.5, 0.0, 0.0)
)

ethene_display_xyz = translated_xyz(
    ethene_atoms,
    ethene_comment,
    translation=(3.0, 0.0, 0.0)
)

In [24]:
reactants_view = py3Dmol.view(
    width=800,
    height=450
)

reactants_view.addModel(
    butadiene_display_xyz,
    "xyz"
)

reactants_view.addModel(
    ethene_display_xyz,
    "xyz"
)

reactants_view.setStyle(
    {},
    {
        "stick": {
            "radius": 0.14
        },
        "sphere": {
            "scale": 0.27
        }
    }
)

reactants_view.setBackgroundColor("white")
reactants_view.zoomTo()

reactants_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [25]:
def atom_position(atoms, atom_number):
    """
    Return the Cartesian coordinates of a one-based atom number.
    """
    atom = atoms[atom_number - 1]

    return np.array(
        [atom["x"], atom["y"], atom["z"]],
        dtype=float
    )


def coordinate_dictionary(atoms, atom_number):
    """
    Return coordinates in the format required by py3Dmol.
    """
    atom = atoms[atom_number - 1]

    return {
        "x": float(atom["x"]),
        "y": float(atom["y"]),
        "z": float(atom["z"]),
    }


def midpoint_dictionary(
    atoms,
    atom_number_1,
    atom_number_2,
    offset=(0.0, 0.0, 0.0),
):
    """
    Return the midpoint between two atoms.
    """
    point_1 = atom_position(atoms, atom_number_1)
    point_2 = atom_position(atoms, atom_number_2)

    midpoint = (point_1 + point_2) / 2.0

    dx, dy, dz = offset
    midpoint = midpoint + np.array([dx, dy, dz], dtype=float)

    return {
        "x": float(midpoint[0]),
        "y": float(midpoint[1]),
        "z": float(midpoint[2]),
    }

In [26]:
def xyz_atom_label(atom):
    """
    Create a label such as C1, C2, H3, H4, etc.
    """
    return f"{atom['element']}{atom['atom_number']}"

In [27]:
def atom_label_position(
    atoms,
    atom_number,
    carbon_offset=0.28,
    hydrogen_offset=0.38,
):
    """
    Move the label slightly outward from the molecular centre
    so that the atom sphere does not hide the label.
    """
    coordinates = np.array(
        [
            [atom["x"], atom["y"], atom["z"]]
            for atom in atoms
        ],
        dtype=float
    )

    molecular_centre = coordinates.mean(axis=0)

    atom = atoms[atom_number - 1]

    atom_coordinates = np.array(
        [atom["x"], atom["y"], atom["z"]],
        dtype=float
    )

    outward_vector = atom_coordinates - molecular_centre
    vector_length = np.linalg.norm(outward_vector)

    if vector_length < 1.0e-12:
        outward_vector = np.array([0.0, 0.0, 1.0], dtype=float)
        vector_length = 1.0

    outward_unit_vector = outward_vector / vector_length

    if atom["element"] == "H":
        offset_distance = hydrogen_offset
    else:
        offset_distance = carbon_offset

    label_coordinates = (
        atom_coordinates
        + offset_distance * outward_unit_vector
    )

    return {
        "x": float(label_coordinates[0]),
        "y": float(label_coordinates[1]),
        "z": float(label_coordinates[2]),
    }

In [28]:
def add_all_atom_labels(
    viewer,
    atoms,
    panel,
    font_size=11,
):
    """
    Add labels for all carbon and hydrogen atoms.
    Carbon labels = black
    Hydrogen labels = blue
    """
    for atom in atoms:
        element = atom["element"]

        if element not in {"C", "H"}:
            continue

        if element == "C":
            font_colour = "black"
        else:
            font_colour = "blue"

        viewer.addLabel(
            xyz_atom_label(atom),
            {
                "position": atom_label_position(
                    atoms,
                    atom["atom_number"]
                ),
                "fontColor": font_colour,
                "backgroundColor": "white",
                "backgroundOpacity": 0.75,
                "fontSize": font_size,
                "showBackground": True,
                "inFront": True,
            },
            viewer=panel
        )

In [29]:
def print_xyz_numbering(atoms, structure_name):
    """
    Print the numbering used in the visualisation.
    """
    print(f"\n{structure_name}")
    print("-" * 45)
    print(
        f"{'XYZ row':>8} "
        f"{'Element':>10} "
        f"{'Displayed label':>17}"
    )
    print("-" * 45)

    for atom in atoms:
        print(
            f"{atom['atom_number']:>8} "
            f"{atom['element']:>10} "
            f"{xyz_atom_label(atom):>17}"
        )

In [30]:
print_xyz_numbering(ethene_atoms, "DARC_ethene")
print_xyz_numbering(butadiene_atoms, "DARC_butadiene")
print_xyz_numbering(product_atoms, "DARC_P1")


DARC_ethene
---------------------------------------------
 XYZ row    Element   Displayed label
---------------------------------------------
       1          C                C1
       2          C                C2
       3          H                H3
       4          H                H4
       5          H                H5
       6          H                H6

DARC_butadiene
---------------------------------------------
 XYZ row    Element   Displayed label
---------------------------------------------
       1          C                C1
       2          C                C2
       3          C                C3
       4          C                C4
       5          H                H5
       6          H                H6
       7          H                H7
       8          H                H8
       9          H                H9
      10          H               H10

DARC_P1
---------------------------------------------
 XYZ row    Element   Displayed label
----------

In [31]:
reaction_numbered_view = py3Dmol.view(
    width=1350,
    height=520,
    viewergrid=(1, 3),
    linked=False
)

# ==========================================================
# PANEL 1: DARC_ethene
# ==========================================================

reaction_numbered_view.addModel(
    ethene_xyz,
    "xyz",
    viewer=(0, 0)
)

reaction_numbered_view.setStyle(
    {},
    {
        "stick": {"radius": 0.13},
        "sphere": {"scale": 0.25}
    },
    viewer=(0, 0)
)

reaction_numbered_view.addLabel(
    "DARC_ethene",
    {
        "position": {"x": 0.0, "y": 0.0, "z": 3.0},
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "black",
        "fontSize": 15,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 0)
)

add_all_atom_labels(
    viewer=reaction_numbered_view,
    atoms=ethene_atoms,
    panel=(0, 0),
    font_size=11
)

reaction_numbered_view.setBackgroundColor(
    "white",
    viewer=(0, 0)
)

reaction_numbered_view.zoomTo(
    viewer=(0, 0)
)

# ==========================================================
# PANEL 2: DARC_butadiene
# ==========================================================

reaction_numbered_view.addModel(
    butadiene_xyz,
    "xyz",
    viewer=(0, 1)
)

reaction_numbered_view.setStyle(
    {},
    {
        "stick": {"radius": 0.13},
        "sphere": {"scale": 0.25}
    },
    viewer=(0, 1)
)

reaction_numbered_view.addLabel(
    "DARC_butadiene",
    {
        "position": {"x": 0.0, "y": 0.0, "z": 3.5},
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "black",
        "fontSize": 15,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 1)
)

add_all_atom_labels(
    viewer=reaction_numbered_view,
    atoms=butadiene_atoms,
    panel=(0, 1),
    font_size=11
)

reaction_numbered_view.setBackgroundColor(
    "white",
    viewer=(0, 1)
)

reaction_numbered_view.zoomTo(
    viewer=(0, 1)
)

# ==========================================================
# PANEL 3: DARC_P1
# ==========================================================

reaction_numbered_view.addModel(
    product_xyz,
    "xyz",
    viewer=(0, 2)
)

reaction_numbered_view.setStyle(
    {},
    {
        "stick": {"radius": 0.13},
        "sphere": {"scale": 0.25}
    },
    viewer=(0, 2)
)

reaction_numbered_view.addLabel(
    "DARC_P1",
    {
        "position": {"x": 0.0, "y": 0.0, "z": 3.8},
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "black",
        "fontSize": 15,
        "showBackground": True,
        "inFront": True,
    },
    viewer=(0, 2)
)

add_all_atom_labels(
    viewer=reaction_numbered_view,
    atoms=product_atoms,
    panel=(0, 2),
    font_size=11
)

reaction_numbered_view.setBackgroundColor(
    "white",
    viewer=(0, 2)
)

reaction_numbered_view.zoomTo(
    viewer=(0, 2)
)

reaction_numbered_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [33]:
product_highlight_numbered_view = py3Dmol.view(
    width=800,
    height=600
)

product_highlight_numbered_view.addModel(
    product_xyz,
    "xyz"
)

product_highlight_numbered_view.setStyle(
    {},
    {
        "stick": {"radius": 0.14},
        "sphere": {"scale": 0.27}
    }
)

# Newly formed bonds
new_bonds = [
    (2, 3),
    (4, 5),
]

for atom_1, atom_2 in new_bonds:
    product_highlight_numbered_view.addCylinder(
        {
            "start": coordinate_dictionary(product_atoms, atom_1),
            "end": coordinate_dictionary(product_atoms, atom_2),
            "radius": 0.055,
            "color": "red",
            "opacity": 0.85,
            "fromCap": True,
            "toCap": True,
        }
    )

# Remaining double bond
product_highlight_numbered_view.addCylinder(
    {
        "start": coordinate_dictionary(product_atoms, 1),
        "end": coordinate_dictionary(product_atoms, 6),
        "radius": 0.060,
        "color": "blue",
        "opacity": 0.85,
        "fromCap": True,
        "toCap": True,
    }
)

product_highlight_numbered_view.addLabel(
    "DARC_P1",
    {
        "position": {"x": 0.0, "y": 0.0, "z": 3.8},
        "backgroundColor": "white",
        "backgroundOpacity": 0.90,
        "fontColor": "black",
        "fontSize": 16,
        "showBackground": True,
        "inFront": True,
    }
)

# Add all atom numbering
add_all_atom_labels(
    viewer=product_highlight_numbered_view,
    atoms=product_atoms,
    panel=None,
    font_size=11
)

product_highlight_numbered_view.setBackgroundColor("white")
product_highlight_numbered_view.zoomTo()

product_highlight_numbered_view

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [34]:
def add_all_atom_labels(
    viewer,
    atoms,
    panel=None,
    font_size=11,
):
    for atom in atoms:
        element = atom["element"]

        if element not in {"C", "H"}:
            continue

        if element == "C":
            font_colour = "black"
        else:
            font_colour = "blue"

        label_style = {
            "position": atom_label_position(
                atoms,
                atom["atom_number"]
            ),
            "fontColor": font_colour,
            "backgroundColor": "white",
            "backgroundOpacity": 0.75,
            "fontSize": font_size,
            "showBackground": True,
            "inFront": True,
        }

        if panel is None:
            viewer.addLabel(
                xyz_atom_label(atom),
                label_style
            )
        else:
            viewer.addLabel(
                xyz_atom_label(atom),
                label_style,
                viewer=panel
            )